In [1]:
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
 
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import TimeSeriesSplit

# 1 : LOAD & VERIFY

In [2]:
df = pd.read_csv(
    "../data/processed/featured/featured_data_final.csv",
    parse_dates=["Date"],
    index_col="Date"
)
df.sort_index(inplace=True)   # Time Series ต้องเรียงตามเวลาเสมอ
 
print(f"Shape          : {df.shape}")
print(f"Date range     : {df.index.min().date()} → {df.index.max().date()}")
print(f"Missing values : {df.isnull().sum().sum()}")
print(f"Index sorted   : {df.index.is_monotonic_increasing}")

Shape          : (2543, 83)
Date range     : 2016-02-12 → 2026-03-27
Missing values : 0
Index sorted   : True


In [3]:
# แยก Feature columns ออกจาก Target columns
feature_cols = [c for c in df.columns if c.startswith("f_")]
target_cols  = ["target_return", "target_direction"]
 
print(f"\nFeature columns : {len(feature_cols)}")
print(f"Target columns  : {target_cols}")
 


Feature columns : 81
Target columns  : ['target_return', 'target_direction']


# 2 : TARGET VARIABLE

#### 2.1 Target Regression (ใช้ target_return ที่สร้างใน ขั้นตอน features)
- target_return = gold_close_ret.shift(-1) = return ของวันพรุ่งนี้

#### 2.2 Target Classification 

In [4]:
# 2.2 Target Classification
# ใช้ผลตอบแทนในอดีต (ถึงแค่วันนี้) มาคำนวณ Threshold (ไม่มี Leakage)
# ใส่ min_periods=1 เพื่อไม่ให้ 60 วันแรกกลายเป็น NaN
threshold = df["f_gold_close_ret"].rolling(60, min_periods=1).std() * 0.5

df["target_direction"] = np.where(
    df["target_return"] > threshold, 1,
    np.where(df["target_return"] < -threshold, -1, 0)
)

# ตรวจผลทันที
print(df["target_direction"].value_counts(normalize=True).sort_index() * 100)

target_direction
-1    25.088478
 0    42.430201
 1    32.481321
Name: proportion, dtype: float64


In [5]:
# threshold = df["target_return"].rolling(60).std() * 0.5

# df["target_direction"] = np.where(
#     df["target_return"] >  threshold,  1,
#     np.where(df["target_return"] < -threshold, -1, 0)
# )

# # ตรวจผลทันที
# print(df["target_direction"].value_counts(normalize=True).sort_index() * 100)

#### 2.3 Distribution ของ Target

In [6]:
print("\nTarget (Regression) stats:")
print(df["target_return"].describe().round(4))
 
print("\nTarget (Classification) distribution:")
cls_counts = df["target_direction"].value_counts().sort_index()
cls_pct    = df["target_direction"].value_counts(normalize=True).sort_index() * 100
for label, name in [(-1, "DOWN"), (0, "SIDE"), (1, "UP")]:
    print(f"  {name:4s} ({label:+d}) : {cls_counts.get(label,0):5d} rows "
          f"({cls_pct.get(label,0):.1f}%)")


Target (Regression) stats:
count    2543.0000
mean        0.0727
std         0.7448
min        -1.9676
25%        -0.3542
50%         0.0326
75%         0.5200
max         2.1235
Name: target_return, dtype: float64

Target (Classification) distribution:
  DOWN (-1) :   638 rows (25.1%)
  SIDE (+0) :  1079 rows (42.4%)
  UP   (+1) :   826 rows (32.5%)


#### 2.4 ตรวจ Class Imbalance

In [7]:
dominant_pct = cls_pct.max()
if dominant_pct > 60:
    print(f"\n  Class Imbalance: dominant class = {dominant_pct:.1f}%")
    print("   แนะนำให้ใช้ class_weight='balanced' หรือ SMOTE")
else:
    print(f"\n Class balance OK (max class = {dominant_pct:.1f}%)")


 Class balance OK (max class = 42.4%)


 # 3 : DATA LEAKAGE FINAL CHECK

กฎ Anti-Leakage:
  - Feature ทุกตัวขึ้นต้นด้วย f_  → ถูก shift(1) แล้วใน ขั้นตอน features
  - target_return = shift(-1)       → เป็นข้อมูลอนาคต ไม่ใช่ปัจจุบัน
  - target_direction คำนวณจาก target_return → ปลอดภัย
  - ห้ามใช้ gold_close_ret วันเดียวกันเป็น Feature
  - ห้าม shuffle ข้อมูลก่อน split

In [8]:
# ตรวจว่า feature ทุกตัวขึ้นต้นด้วย f_
non_shifted = [c for c in feature_cols if not c.startswith("f_")]
if non_shifted:
    print(f"Features ที่ยังไม่ได้ shift: {non_shifted}")
else:
    print("Feature ทุกตัวถูก shift(1) แล้ว (ขึ้นต้นด้วย f_)")

Feature ทุกตัวถูก shift(1) แล้ว (ขึ้นต้นด้วย f_)


In [9]:
# ตรวจ index ยังเรียงตามเวลา
assert df.index.is_monotonic_increasing, "Index ไม่ได้เรียงตามเวลา!"
print("Index เรียงตามเวลา (Monotonic Increasing)")
 

Index เรียงตามเวลา (Monotonic Increasing)


In [10]:
# ตรวจไม่มี NaN ใน feature และ target
nan_count = df[feature_cols + target_cols].isnull().sum().sum()
print(f"NaN count = {nan_count}")

NaN count = 0


# 4 : TRAIN / VAL / TEST SPLIT (TIME-BASED)

ใช้เป็น Time-based Split (ไม่ใช่ Random) เนื่องจาก

Financial Time Series มีคุณสมบัติ:
  1. Autocorrelation : วันนี้มีความสัมพันธ์กับเมื่อวาน
  2. Regime Shift    : ตลาดปี 2024-2026 ต่างจาก 2015-2023
  3. Look-ahead Bias : ถ้า shuffle แล้ว test set อาจอยู่ก่อน train set
 
ถ้า Random Split:
  → โมเดลอาจ "เห็น" ข้อมูลปี 2025 ตอน train แล้วถูกทดสอบด้วยปี 2020
  → Metric ดีเกินจริง (Overly Optimistic)
 
Strategy ที่เลือก: Regime-Aware Split
  Train : 2016-02 → 2021-12 (ช่วงปกติ pre-inflation)
  Val   : 2022-01 → 2023-12 (ช่วง rate hike cycle)
  Test  : 2024-01 → 2026-03 (Regime Shift ชัดเจน)
 
เหตุผล:
  - Train ด้วยข้อมูล ปกติ ก่อน
  - Val ใช้ปรับ hyperparameter ในช่วง transition
  - Test ใช้ประเมิน robustness กับ Regime ใหม่จริงๆ
  - ถ้าโมเดล generalize ได้ดีใน Test → เชื่อถือได้สูง

In [11]:
TRAIN_END = "2021-12-31"
VAL_END   = "2023-12-31"
# Test = 2024-01-01 → end
 
train_df = df[df.index <= TRAIN_END]
val_df   = df[(df.index > TRAIN_END) & (df.index <= VAL_END)]
test_df  = df[df.index > VAL_END]

In [12]:
# ── แยก X และ y
X_train = train_df[feature_cols]
X_val   = val_df[feature_cols]
X_test  = test_df[feature_cols]

##### Regression targets

In [13]:
# Regression targets
y_train_reg = train_df["target_return"]
y_val_reg   = val_df["target_return"]
y_test_reg  = test_df["target_return"]
 

##### Classification targets

In [14]:
# Classification targets
y_train_clf = train_df["target_direction"]
y_val_clf   = val_df["target_direction"]
y_test_clf  = test_df["target_direction"]

In [15]:
# Summary
total = len(df)
for name, subset in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    pct = len(subset) / total * 100
    print(f"  {name:5s}: {subset.index.min().date()} → {subset.index.max().date()} "
          f"| {len(subset):5d} rows ({pct:.1f}%)")
 
print(f"\n  Total : {total} rows (100%)")

  Train: 2016-02-12 → 2021-12-31 |  1479 rows (58.2%)
  Val  : 2022-01-03 → 2023-12-29 |   501 rows (19.7%)
  Test : 2024-01-02 → 2026-03-27 |   563 rows (22.1%)

  Total : 2543 rows (100%)


In [16]:
# ตรวจ Classification Target Distribution แยกตาม split
print("Classification Target Distribution per Split:")
print(f"{'Label':<10} {'Train':>10} {'Val':>10} {'Test':>10}")

for label, name in [(-1, "DOWN"), (0, "SIDE"), (1, "UP")]:
    t  = (y_train_clf == label).mean() * 100
    v  = (y_val_clf   == label).mean() * 100
    te = (y_test_clf  == label).mean() * 100
    print(f"  {name:<8} {t:>9.1f}% {v:>9.1f}% {te:>9.1f}%")

Classification Target Distribution per Split:
Label           Train        Val       Test
  DOWN          25.4%      28.5%      21.3%
  SIDE          43.9%      40.5%      40.3%
  UP            30.8%      30.9%      38.4%


In [17]:
# Debug ก่อน assert
print(f"Data range   : {df.index.min().date()} → {df.index.max().date()}")
print(f"Train end    : {train_df.index.max().date()}")
print(f"Val start    : {val_df.index.min().date()}")
print(f"Val end      : {val_df.index.max().date()}")
print(f"Test start   : {test_df.index.min().date()}")
print(f"Train rows   : {len(train_df)}")
print(f"Val rows     : {len(val_df)}")
print(f"Test rows    : {len(test_df)}")

Data range   : 2016-02-12 → 2026-03-27
Train end    : 2021-12-31
Val start    : 2022-01-03
Val end      : 2023-12-29
Test start   : 2024-01-02
Train rows   : 1479
Val rows     : 501
Test rows    : 563


In [18]:
# ตรวจ Time Order ไม่ซ้อนทับกัน
assert train_df.index.max() < val_df.index.min(),   "Train/Val overlap!"
assert val_df.index.max()   < test_df.index.min(),  "Val/Test overlap!"
print("\nไม่มีการซ้อนทับระหว่าง Train/Val/Test")


ไม่มีการซ้อนทับระหว่าง Train/Val/Test


# 5 : SCALING (RobustScaler)

ทำไม RobustScaler? 

Financial data มี Black Swan Events (เช่น COVID-19 ปี 2020)
ที่ทำให้ค่า Return สวิงถึง ±10% ในวันเดียว
 
StandardScaler คำนวณ mean & std รวม outlier → scale ผิดเพี้ยน
MinMaxScaler ยิ่งแย่กว่า → outlier กดค่าปกติให้แบนราบ
RobustScaler ใช้ median & IQR → ไม่ถูก outlier รบกวน
 
กฎสำคัญ: fit เฉพาะ Train เท่านั้น
  → ป้องกัน "Statistical Leakage" (โมเดลรู้ distribution ของ val/test ล่วงหน้า)

In [19]:
scaler = RobustScaler()

In [20]:
# Fit บน Train เท่านั้น
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val) # transform เท่านั้น
X_test_scaled = scaler.transform(X_test) # transform เท่านั้น

# แปลงกลับเป็น DataFrame เพื่อให้ยังมี column names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols, index=X_train.index)
X_val_scaled   = pd.DataFrame(X_val_scaled,   columns=feature_cols, index=X_val.index)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=feature_cols, index=X_test.index)

print(f"Scaler fitted on Train only: {X_train.shape[0]} rows")
print(f"X_train_scaled : {X_train_scaled.shape}")
print(f"X_val_scaled : {X_val_scaled.shape}")
print(f"X_test_scaled : {X_test_scaled.shape}")

Scaler fitted on Train only: 1479 rows
X_train_scaled : (1479, 81)
X_val_scaled : (501, 81)
X_test_scaled : (563, 81)


In [21]:
# ================================================================================
# 5.5 : FEATURE SELECTION — ตัด zero-importance & high-correlation features
# ================================================================================
print("\n5.5 : Feature Selection...")

from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

# ── Step 1: Quick RF เพื่อหา importance
rf_quick = RandomForestRegressor(
    n_estimators=100,
    max_depth=6,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)
rf_quick.fit(X_train_scaled, y_train_reg)

feat_imp = pd.Series(
    rf_quick.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

# ── Step 2: ตัด zero-importance features
zero_imp_cols = feat_imp[feat_imp == 0].index.tolist()
print(f"\n   Features ที่มี importance = 0 ({len(zero_imp_cols)} ตัว):")
print(f"   {zero_imp_cols}")

# ── Step 3: ตัด high-correlation features (r > 0.95)
corr_matrix = X_train_scaled.corr().abs()
upper_tri   = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_cols = [
    col for col in upper_tri.columns 
    if any(upper_tri[col] > 0.95)
]
print(f"\n   Features ที่ correlate กันสูง >0.95 ({len(high_corr_cols)} ตัว):")
print(f"   {high_corr_cols}")

# ── Step 4: รวม features ที่จะตัดออก
# ตัดเฉพาะ zero importance ก่อน (conservative approach)
# high_corr ดูก่อนว่าสำคัญไหม
cols_to_drop = zero_imp_cols
print(f"\n   Features ที่จะตัดออก: {len(cols_to_drop)} ตัว")

# ── Step 5: กรอง X sets
selected_cols = [c for c in feature_cols if c not in cols_to_drop]

X_train_scaled = X_train_scaled[selected_cols]
X_val_scaled   = X_val_scaled[selected_cols]
X_test_scaled  = X_test_scaled[selected_cols]

print(f"\n   Features ก่อน selection : {len(feature_cols)}")
print(f"   Features หลัง selection : {len(selected_cols)}")
print(f"   X_train shape : {X_train_scaled.shape}")
print(f"   X_val shape   : {X_val_scaled.shape}")
print(f"   X_test shape  : {X_test_scaled.shape}")

# ── Step 6: บันทึก selected feature list
pd.Series(selected_cols).to_csv(
    "../data/processed/splits/selected_features.csv",
    index=False, header=["feature"]
)
print(f"\n   ✔ Saved selected features list")


5.5 : Feature Selection...



   Features ที่มี importance = 0 (12 ตัว):
   ['f_gold_momentum_regime', 'f_is_q1', 'f_is_q4', 'f_dxy_above_ma200', 'f_real_rate_negative', 'f_regime_high_vol', 'f_vix_spike', 'f_vix_regime', 'f_high_inflation', 'f_inverted_curve', 'f_days_gap', 'f_gold_vol_regime']

   Features ที่ correlate กันสูง >0.95 (5 ตัว):
   ['f_vol_spread_vix_gold', 'f_log_gold_vol_30d', 'f_quarter', 'f_vix_close', 'f_yield_close']

   Features ที่จะตัดออก: 12 ตัว

   Features ก่อน selection : 81
   Features หลัง selection : 69
   X_train shape : (1479, 69)
   X_val shape   : (501, 69)
   X_test shape  : (563, 69)

   ✔ Saved selected features list


# 6 : SANITY CHECK

In [22]:
checks = {
    "Train shape OK"     : X_train_scaled.shape[1] == len(selected_cols),
    "Val shape OK"       : X_val_scaled.shape[1] == len(selected_cols),
    "Test shape OK"      : X_test_scaled.shape[1] == len(selected_cols),
    "Train no NaN"         : X_train_scaled.isnull().sum().sum() == 0,
    "Val no NaN"           : X_val_scaled.isnull().sum().sum() == 0,
    "Test no NaN"          : X_test_scaled.isnull().sum().sum() == 0,
    "Train index sorted"   : X_train_scaled.index.is_monotonic_increasing,
    "Val index sorted"     : X_val_scaled.index.is_monotonic_increasing,
    "Test index sorted"    : X_test_scaled.index.is_monotonic_increasing,
    "No Train/Val overlap" : X_train_scaled.index.max() < X_val_scaled.index.min(),
    "No Val/Test overlap"  : X_val_scaled.index.max()   < X_test_scaled.index.min(),
}

In [23]:
all_pass = True
for check_name, result in checks.items():
    icon = "ผ่าน" if result else "ไม่ผ่าน"
    print(f"  {icon} {check_name}")
    if not result:
        all_pass = False
 
print(f"\n{'ALL CHECKS PASSED' if all_pass else 'SOME CHECKS FAILED'}")

  ผ่าน Train shape OK
  ผ่าน Val shape OK
  ผ่าน Test shape OK
  ผ่าน Train no NaN
  ผ่าน Val no NaN
  ผ่าน Test no NaN
  ผ่าน Train index sorted
  ผ่าน Val index sorted
  ผ่าน Test index sorted
  ผ่าน No Train/Val overlap
  ผ่าน No Val/Test overlap

ALL CHECKS PASSED


In [24]:
# ── Target distribution per split
print("\nClassification Target Distribution:")
print(f"{'Label':<10} {'Train':>10} {'Val':>10} {'Test':>10}")
for label, name in [(-1, "DOWN"), (0, "SIDE"), (1, "UP")]:
    t = (y_train_clf == label).mean() * 100
    v = (y_val_clf   == label).mean() * 100
    te= (y_test_clf  == label).mean() * 100
    print(f"  {name:<8} {t:>9.1f}% {v:>9.1f}% {te:>9.1f}%")
 


Classification Target Distribution:
Label           Train        Val       Test
  DOWN          25.4%      28.5%      21.3%
  SIDE          43.9%      40.5%      40.3%
  UP            30.8%      30.9%      38.4%


In [25]:
# ── Regression Target Stats per split
print("\nRegression Target Stats (next-day return %):")
print(f"{'Metric':<10} {'Train':>10} {'Val':>10} {'Test':>10}")
for metric in ["mean", "std", "min", "max"]:
    t  = getattr(y_train_reg, metric)()
    v  = getattr(y_val_reg,   metric)()
    te = getattr(y_test_reg,  metric)()
    print(f"  {metric:<8} {t:>10.4f} {v:>10.4f} {te:>10.4f}")


Regression Target Stats (next-day return %):
Metric          Train        Val       Test
  mean         0.0416     0.0206     0.2008
  std          0.7022     0.7650     0.8188
  min         -1.9661    -1.9676    -1.9389
  max          2.0649     2.1235     2.1235


# 7 : TIMESERIESSPLIT / WALK-FORWARD CV

Walk-Forward Cross-Validation:
แทนที่จะ validate ครั้งเดียว ให้เลื่อน window ไปเรื่อยๆ:
 
  Fold 1: Train [0→400]   Val [400→480]
  Fold 2: Train [0→480]   Val [480→560]
  Fold 3: Train [0→560]   Val [560→640]
  ...
 
ข้อดี:
  - ได้ metric หลาย fold → เชื่อถือกว่า single split
  - เห็นว่าโมเดล stable ข้ามช่วงเวลาหรือไม่
  - ใช้ข้อมูลได้มากขึ้นในการ train
 
ใช้กับ Train set เท่านั้น (ไม่ยุ่งกับ Test set ระหว่าง tune)

#### 7.1 TimeSeriesSplit บน Train set

In [26]:
N_SPLITS = 5
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
 
print(f"\nTimeSeriesSplit: {N_SPLITS} folds บน Train set ({len(X_train)} rows)")
print(f"{'Fold':<6} {'Train rows':>12} {'Val rows':>10} {'Train period':<25} {'Val period'}")
print("-" * 80)
 
for fold, (tr_idx, vl_idx) in enumerate(tscv.split(X_train_scaled), 1):
    tr_dates = X_train_scaled.index[tr_idx]
    vl_dates = X_train_scaled.index[vl_idx]
    print(f"  {fold:<4} {len(tr_idx):>12,} {len(vl_idx):>10,} "
          f"  {str(tr_dates.min().date())+'→'+str(tr_dates.max().date()):<25} "
          f"  {vl_dates.min().date()}→{vl_dates.max().date()}")
 


TimeSeriesSplit: 5 folds บน Train set (1479 rows)
Fold     Train rows   Val rows Train period              Val period
--------------------------------------------------------------------------------
  1             249        246   2016-02-12→2017-02-09       2017-02-10→2018-02-02
  2             495        246   2016-02-12→2018-02-02       2018-02-05→2019-01-28
  3             741        246   2016-02-12→2019-01-28       2019-01-29→2020-01-17
  4             987        246   2016-02-12→2020-01-17       2020-01-21→2021-01-11
  5           1,233        246   2016-02-12→2021-01-11       2021-01-12→2021-12-31


#### 7.2 Walk-Forward Evaluation Function (พร้อมใช้)

In [27]:
def walk_forward_evaluate(model, X, y, n_splits=5, verbose=True):
    """
    Walk-forward cross-validation สำหรับ Time Series
    
    Parameters:
        model   : sklearn estimator (fit/predict)
        X       : feature DataFrame (scaled, sorted by time)
        y       : target Series
        n_splits: จำนวน fold
    
    Returns:
        scores  : dict ของ metric แต่ละ fold
    """
    from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                                  accuracy_score, f1_score)
 
    tscv   = TimeSeriesSplit(n_splits=n_splits)
    scores = {"rmse": [], "mae": [], "fold_dates": []}
    
    for fold, (tr_idx, vl_idx) in enumerate(tscv.split(X), 1):
        X_tr, X_vl = X.iloc[tr_idx], X.iloc[vl_idx]
        y_tr, y_vl = y.iloc[tr_idx], y.iloc[vl_idx]
        
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_vl)
        
        rmse = np.sqrt(mean_squared_error(y_vl, y_pred))
        mae  = mean_absolute_error(y_vl, y_pred)
        
        scores["rmse"].append(rmse)
        scores["mae"].append(mae)
        scores["fold_dates"].append(
            (X_vl.index.min().date(), X_vl.index.max().date())
        )
        
        if verbose:
            print(f"  Fold {fold}: RMSE={rmse:.4f}  MAE={mae:.4f}  "
                  f"Val={X_vl.index.min().date()}→{X_vl.index.max().date()}")
    
    print(f"\n  Mean RMSE: {np.mean(scores['rmse']):.4f} ± {np.std(scores['rmse']):.4f}")
    print(f"  Mean MAE : {np.mean(scores['mae']):.4f} ± {np.std(scores['mae']):.4f}")
    return scores
 
print("\n walk_forward_evaluate() พร้อมใช้งาน")
print("ตัวอย่าง: scores = walk_forward_evaluate(rf_model, X_train_scaled, y_train_reg)")


 walk_forward_evaluate() พร้อมใช้งาน
ตัวอย่าง: scores = walk_forward_evaluate(rf_model, X_train_scaled, y_train_reg)


# 8. SAVE

In [28]:
import os
 
# บันทึก scaled features
X_train_scaled.to_csv("../data/processed/splits/X_train.csv")
X_val_scaled.to_csv("../data/processed/splits/X_val.csv")
X_test_scaled.to_csv("../data/processed/splits/X_test.csv")
 
# บันทึก targets
y_train_reg.to_csv("../data/processed/splits/y_train_reg.csv")
y_val_reg.to_csv("../data/processed/splits/y_val_reg.csv")
y_test_reg.to_csv("../data/processed/splits/y_test_reg.csv")
 
y_train_clf.to_csv("../data/processed/splits/y_train_clf.csv")
y_val_clf.to_csv("../data/processed/splits/y_val_clf.csv")
y_test_clf.to_csv("../data/processed/splits/y_test_clf.csv")
 
print("Saved files:")
files = [
    "X_train.csv", "X_val.csv", "X_test.csv",
    "y_train_reg.csv", "y_val_reg.csv", "y_test_reg.csv",
    "y_train_clf.csv", "y_val_clf.csv", "y_test_clf.csv",
]
for f in files:
    print(f"   ../data/processed/splits/{f}")

Saved files:
   ../data/processed/splits/X_train.csv
   ../data/processed/splits/X_val.csv
   ../data/processed/splits/X_test.csv
   ../data/processed/splits/y_train_reg.csv
   ../data/processed/splits/y_val_reg.csv
   ../data/processed/splits/y_test_reg.csv
   ../data/processed/splits/y_train_clf.csv
   ../data/processed/splits/y_val_clf.csv
   ../data/processed/splits/y_test_clf.csv


In [29]:
# ── Final Summary
print("\n" + "=" * 60)
print("  FINAL SUMMARY")
print("=" * 60)
print(f"  Features            : {len(feature_cols)}")
print(f"  X_train             : {X_train_scaled.shape}  "
      f"({X_train_scaled.index.min().date()} → {X_train_scaled.index.max().date()})")
print(f"  X_val               : {X_val_scaled.shape}  "
      f"({X_val_scaled.index.min().date()} → {X_val_scaled.index.max().date()})")
print(f"  X_test              : {X_test_scaled.shape}  "
      f"({X_test_scaled.index.min().date()} → {X_test_scaled.index.max().date()})")
print(f"  Scaler              : RobustScaler (fit on Train only)")
print(f"  CV Strategy         : TimeSeriesSplit ({N_SPLITS} folds)")
print(f"  Target (Regression) : next-day % return")
print(f"  Target (Classif.)   : -1/0/+1 (Dynamic Threshold)")


  FINAL SUMMARY
  Features            : 81
  X_train             : (1479, 69)  (2016-02-12 → 2021-12-31)
  X_val               : (501, 69)  (2022-01-03 → 2023-12-29)
  X_test              : (563, 69)  (2024-01-02 → 2026-03-27)
  Scaler              : RobustScaler (fit on Train only)
  CV Strategy         : TimeSeriesSplit (5 folds)
  Target (Regression) : next-day % return
  Target (Classif.)   : -1/0/+1 (Dynamic Threshold)
